In [ ]:
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import pandas as pd
from collections import deque

In [2]:
random.seed(42)

In [4]:
with open("./../data/new_data_new_labeling.json", "r", encoding="utf-8") as f:
    combined_conversations = json.load(f)

In [5]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer('all-mpnet-base-v2') 

d:\anaconda\envs\torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4911.74it/s]


In [7]:
src_embedded = []

for conv_id, conversation in enumerate(combined_conversations):
    convo = []    
    for turn_id, turn in enumerate(conversation):
        turn_embedded = {} 
        u_emb = emb_model.encode(turn["u"])
        a_emb = emb_model.encode(turn["a"])
        turn_embedded["u"] = u_emb
        turn_embedded["a"] = a_emb
        turn_embedded["label"] = turn["label"]
        turn_embedded["conv_id"] = conv_id
        turn_embedded["turn_id"] = turn_id
        convo.append(turn_embedded)
        
    src_embedded.append(convo)

# Model

In [8]:
class stateSpaceModel(nn.Module):
    def __init__(self,state_dim, input_dim, hidden_dim1,hidden_dim2, output_dim):
        super(stateSpaceModel, self).__init__()
        # F(x_{t-1},u_t)=>x_t (Transition model)
        self.Fxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, state_dim)
        )

        # G(x_t,u_t)=>z_t (Observation model)
        self.Gxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, output_dim)
        )
        
    def forward(self, x_prev, u):
        # gets xt
        ux_prev = torch.cat([u,x_prev], dim=-1)
        x_curr = self.Fxu(ux_prev)

        # gets zt
        ux_curr = torch.cat([u,x_curr], dim=-1)
        zt = self.Gxu(ux_curr)
        return x_curr, zt

In [9]:
state_dim = 768    # x_t
input_dim = 768    # u_t
output_dim = 768   # z_t
hidden_dim_ssm1 = 1200  
hidden_dim_ssm2 = 900  

model=stateSpaceModel(state_dim=state_dim, input_dim=input_dim, hidden_dim1=hidden_dim_ssm1, \
                      hidden_dim2=hidden_dim_ssm2, output_dim=output_dim)

In [10]:
checkpoint = torch.load("./../models/models_best_ssm_new_data.pth", map_location=torch.device('cpu'))
model.load_state_dict(checkpoint["ssm"])
model.eval()

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_21396\454073132.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./../models/models_best_ssm_

stateSpaceModel(
  (Fxu): Sequential(
    (0): Linear(in_features=1536, out_features=1200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1200, out_features=900, bias=True)
    (3): ReLU()
    (4): Linear(in_features=900, out_features=768, bias=True)
  )
  (Gxu): Sequential(
    (0): Linear(in_features=1536, out_features=1200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1200, out_features=900, bias=True)
    (3): ReLU()
    (4): Linear(in_features=900, out_features=768, bias=True)
  )
)

In [11]:
src_convos=[]
with torch.no_grad(): 
    for convo in src_embedded:
        single_conversation=[]
        x_prev = torch.zeros(state_dim)
        for turn in convo:
            u = torch.tensor(turn['u'], dtype=torch.float32)
            x_curr, zt = model(x_prev, u)
            single_conversation.append({"x_t":x_curr,
                                        "ut":u,
                                        "zt_approx":zt,
                                        "y":turn['label'],
                                        "conv_id":turn["conv_id"],
                                        "turn_id":turn["turn_id"],
                                        })
            x_prev = x_curr
        src_convos.append(single_conversation)

In [12]:
torch.save(src_convos, "./../data/all_conversations_new_data_new_label.pt")

In [13]:
src_convos=torch.load("./../data/all_conversations_new_data_new_label.pt")

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_21396\316870234.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  src_convos=torch.load("./../data/all_conversations_ne

In [ ]:
def feature_engineering(x_t,u_t,x_prev,x_prev_4step_back=None):
    features = []

    state_drift = torch.norm(x_t -x_prev).item()
    features.append(state_drift)

    state_input_distance = torch.norm(x_t -u_t).item()
    features.append(state_input_distance)
        
    long_term_state_drift = torch.norm(x_t - x_prev_4step_back).item()
    features.append(long_term_state_drift)

    state_similarity = F.cosine_similarity(x_t.unsqueeze(0),x_prev.unsqueeze(0)).item()
    features.append(state_similarity)

    state_input_similarity = F.cosine_similarity(x_t.unsqueeze(0),u_t.unsqueeze(0)).item()
    features.append(state_input_similarity)

    long_term_state_similarity = F.cosine_similarity(x_t.unsqueeze(0),x_prev_4step_back.unsqueeze(0)).item()
    features.append(long_term_state_similarity)

    features.append(torch.mean((x_t - x_prev) ** 2).item())
    features.append(torch.mean((x_t - x_prev_4step_back) ** 2).item())

    return torch.tensor(features, dtype=torch.float32)

In [16]:
X_vecs = []
X_features = []
y = []

for convo in src_convos:
    first_x = convo[0]["x_t"]
    first_u = convo[0]["ut"]

    x_prev = torch.zeros_like(convo[0]["x_t"])
    x_history = deque(maxlen=4)

    for turn in convo:
        X_vecs.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))

        if len(x_history) == 4:
            x_prev_4step_back = x_history[0]
        else:
            x_prev_4step_back = first_x

        handcrafted = feature_engineering(turn["x_t"],turn["ut"],x_prev,x_prev_4step_back).numpy()
        ids = np.array([turn["conv_id"], turn["turn_id"]],dtype=np.int32)
        handcrafted = np.concatenate([handcrafted, ids])

        X_features.append(handcrafted)
        y.append(turn["y"])

        x_history.append(turn["x_t"])
        x_prev = turn["x_t"]

In [ ]:
X_vecs_np = torch.stack(X_vecs).cpu().numpy()

feature_columns = [
    "state_drift", "state_input_distance", "long_term_state_drift",
    "state_similarity", "state_input_similarity",
    "long_term_state_similarity", 
    "mean_squared_change_short",
    "mean_squared_change_long",
    "conv_id", 
    "turn_id",
]

X_features_df = pd.DataFrame(X_features, columns=feature_columns)

df = pd.concat([
    pd.DataFrame(X_vecs_np),
    X_features_df
], axis=1)

df["y"] = y

In [21]:
df.to_csv("./../data/ssm_all_features_new_data_new_label.csv", index=False)